In [ ]:
# Install required libraries
!pip install -q transformers torch pandas

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load GPT-2 tokenizer and model
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

# Function to calculate top-k next-token probabilities
def top_k_next(prompt, k=5):
    ids = tok(prompt, return_tensors="pt").input_ids

    with torch.no_grad():
        logits = model(ids).logits[0, -1]

    probs = torch.softmax(logits, dim=-1)
    p, idx = probs.topk(k)

    return [
        {
            "prompt": prompt,
            "rank": r + 1,
            "token": repr(tok.decode(i)),
            "token_id": int(i),
            "logit": round(float(logits[i]), 2),
            "probability": round(float(pp), 4)
        }
        for r, (pp, i) in enumerate(zip(p, idx))
    ]

# Input prompts
prompts = [
    "The capital of India is",
    "Machine learning is a subset of",
    "I drank a cup of",
    "The opposite of hot is"
]

# Generate probability table
table = pd.DataFrame([
    row
    for pr in prompts
    for row in top_k_next(pr)
])

print("Probability table (top-5 next tokens):")
display(table)

# Autoregressive generation (greedy)
# Predict one token at a time

ids = tok("Generative AI is", return_tensors="pt").input_ids

print("\nAutoregressive Text Generation:")

for step in range(8):
    with torch.no_grad():
        nxt = model(ids).logits[0, -1].argmax()

    ids = torch.cat([ids, nxt.view(1, 1)], dim=1)

    print(f"Step {step + 1}: {tok.decode(ids[0])!r}")

print("\nExperiment 7 completed successfully!")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]